# Compute HelpSteer2 M1 and C1 Coefficients

This notebook computes five-objective merge coefficient vectors from the saved HelpSteer2 relationship matrix $R$.

It evaluates three coefficient-selection methods:

- **Direct preference:** $\lambda = p$
- **M1:** relationship-softmax correction controlled by $\tau$
- **C1:** CAGrad-inspired one-shot correction controlled by $\rho$

The thesis pipeline is:

$$\delta_i \rightarrow R \rightarrow \lambda = f(p, R) \rightarrow \theta(\lambda)$$

The notebook writes a coefficient table and compact method metadata under `results/`.

## Clone or update the repository

The following cell starts from `/content`. It updates an existing valid repository or clones a fresh copy, avoiding nested folders such as `/content/master-thesis/master-thesis`.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")

## Show the repository structure

Confirm that the coefficient script and HelpSteer2 relationship-matrix result are available.

In [ ]:
!pwd
!ls
!ls scripts
!ls results

## Install lightweight dependencies

NumPy provides the matrix calculations, SciPy provides the SLSQP optimizer used by C1, and pandas provides readable table previews.

In [ ]:
!pip install -q numpy scipy pandas

In [ ]:
import json
import numpy as np
import pandas as pd
import scipy

print("NumPy version:", np.__version__)
print("SciPy version:", scipy.__version__)
print("pandas version:", pd.__version__)

## Check the relationship matrix

The coefficient computation uses `results/helpsteer2_relationship_matrix.csv`. Notebook 11 creates this file from the five objective-specific LoRA adapters.

In [ ]:
from pathlib import Path

matrix_path = Path("results/helpsteer2_relationship_matrix.csv")

if not matrix_path.is_file():
    raise FileNotFoundError(
        f"Missing relationship matrix: {matrix_path}. "
        "Run Notebook 11 or scripts/compute_helpsteer2_relationship_matrix.py first."
    )

print(f"Relationship matrix found: {matrix_path}")

## Preview the relationship matrix

Rows and columns follow this objective order: helpfulness, correctness, coherence, complexity, and verbosity.

In [ ]:
relationship_df = pd.read_csv(matrix_path, index_col="adapter")
relationship_df

## Compute direct-preference, M1, and C1 coefficients

The script evaluates four example preference vectors. M1 uses $\tau \in \{0.5, 1.0, 2.0\}$, while C1 uses $\rho \in \{0.1, 1.0, 10.0\}$. The direct-preference row provides $\lambda = p$ for each preference.

In [ ]:
!python scripts/compute_helpsteer2_m1_c1_coefficients.py

## Inspect the output files

The script creates:

- `results/helpsteer2_m1_c1_coefficients.csv`
- `results/helpsteer2_m1_c1_coefficients_metadata.json`

In [ ]:
!ls results

In [ ]:
coefficients_path = Path("results/helpsteer2_m1_c1_coefficients.csv")
coefficients_df = pd.read_csv(coefficients_path)

print(f"Rows: {len(coefficients_df)}")
display(coefficients_df.head(10))

In [ ]:
metadata_path = Path(
    "results/helpsteer2_m1_c1_coefficients_metadata.json"
)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print(json.dumps(metadata, indent=2))

## Understand the main columns

- `preference_name` identifies the example user preference vector $p$.
- `method` identifies `direct_preference`, `M1`, or `C1`.
- `hyperparameter_name` and `hyperparameter_value` record $\tau$ for M1 or $\rho$ for C1.
- `p_*` columns contain the original preference vector.
- `lambda_*` columns contain the computed merge coefficients.
- `score_*` columns contain the relationship scores $R\lambda$.
- `min_relationship_score` is the smallest component of $R\lambda$, representing the worst relationship alignment across the five objectives.
- `l1_distance_to_p` and `l2_distance_to_p` measure how far the computed coefficients moved from the original preference vector.

## Compare methods for one preference

This optional view makes it easier to compare the coefficient vectors and distances for the `quality_focused` preference.

In [ ]:
comparison_columns = [
    "method",
    "hyperparameter_name",
    "hyperparameter_value",
    "lambda_helpfulness",
    "lambda_correctness",
    "lambda_coherence",
    "lambda_complexity",
    "lambda_verbosity",
    "min_relationship_score",
    "l1_distance_to_p",
    "l2_distance_to_p",
]

quality_comparison = coefficients_df.loc[
    coefficients_df["preference_name"] == "quality_focused",
    comparison_columns,
]
quality_comparison

## Git safety check

The small CSV and JSON result files are suitable for version control.

Do not commit:

- `adapters/`
- ZIP backup files
- `.safetensors` or `.bin` files
- checkpoints or full model files

In [ ]:
!git status

## What this notebook establishes

This notebook produces reproducible direct-preference, M1, and C1 coefficient vectors from the five-objective HelpSteer2 relationship matrix and records their relationship scores and distances from the original preferences.